###### Content under Creative Commons Attribution license CC-BY 4.0, code under BSD 3-Clause License © 2022  by D. Koehn, T. Meier and J. Stampa, notebook style sheet by L.A. Barba, N.C. Clementi

# Digital Signal Processing in Geophysics 

## Chapter 4: Data Pre-Processing

### 4.2 Effect of Data Gaps on the Signal Spectrum

First, we will examine how the spectrum of a signal changes when missing measurement intervals are filled using different methods. A simple way to replace missing measurement values is to set the corresponding interval to zero. This corresponds to multiplying the original, gap-free function $f(t)$ by a boxcar function. The result is the function $\tilde{f}(t)$, where $2T_e$ is the width of the missing interval and $t_e$ is the center of the gap:

\begin{equation}
  \tilde{f}(t) = f(t) \, (1-boxcar{(t-t_e,T_e)}) \,.
\end{equation}

For the spectrum of the signal $\tilde{F}(\omega)$, we have:

\begin{align}
  \tilde{F}(\omega) &= F(\omega) * (\delta(\omega) - 2 T_e sinc{(T_e \omega)} e^{-i\omega t_e})\, ,\notag \\
  & = F(\omega) - F(\omega) * 2 T_e sinc{(T_e \omega)} e^{-i\omega t_e} \, . 
\end{align}

It turns out that the spectrum of the data series with gaps consists of the actual spectrum of the undisturbed data series and a noise term. The disturbance term consists of the original spectrum convolved with a sinc function. In particular, this smears the peaks of the spectrum and generates secondary maxima. As the width ($T_e$) of the missing interval decreases, the spectrum approaches the undisturbed spectrum.

We will examine this in more detail using a simple example in the following Jupyter Notebook.

In [ ]:
# Importiere Python Bibliotheken 
# ------------------------------
#%matplotlib inline
from ipywidgets import interactive
import matplotlib.pyplot as plt
import numpy as np

In addition to the standard Python libraries `NumPy` and `Matplotlib`, this time we’ll also import `ipywidgets`, which allows us to create interactive notebook cells so we can more easily experiment with the parameters.
 
Based on the Python code from Exercise 3.4, I have written a small function that inserts a data gap into a 1000-second-long time series using a sine function. The input parameters are the width of the gap $T_e$ and the temporal center of the gap $t_e$.

In [ ]:
def data_gaps(Te, te):
    
    # Define parameters
    T = 25.                  # period 1 [s]
    dt = .1                  # time sampling [s]
    L = 1000.                # length of the time series [s]

    # Define sine function ...
    omega = 2. * np.pi / T   # compute circular frequency from period
    t = np.arange(0,L+dt,dt) # compute time vector
    x = np.sin(omega*t)      # compute sine wave
    
    # ... add local high frequency signal ...
    #T1 = 15                    # period 2 [s]
    #omega1 = 2. * np.pi / T1   # compute circular frequency from period 2
    #x[1000:4000] = np.sin(omega*t[1000:4000]) + np.sin(omega1*t[1000:4000])
    
    # Add data gap 
    NTeh = (int)((Te/2) // dt) # half size of the gap [#samples]
    Nte = (int)(te // dt)      # location of the gap [#samples]
    
    # Set values inside the gap to zero
    x[Nte-NTeh:Nte+NTeh] = 0.
    
    # Fourier transform
    X = np.fft.fft(x)
    
    # Normalize by length of the time series
    N = len(x)               # length of the time series [#samples]
    X = X / (N*dt)
    
    # estimate frequencies    
    freq = np.fft.fftfreq(N, d=dt)
    
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    plt.subplot(121)
    
    plt.plot(t, x, 'b')
    plt.plot(t[Nte],0,'ro')   # mark center of data gap
    plt.xlabel('Time (s)')
    plt.ylabel('Sin(t) ( )')
    plt.title('Time Series Sin(t)')
    
    # Plot Spectrum
    plt.subplot(122)
    
    plt.plot(freq, np.abs(X), 'b')
    plt.xlabel('Freq (Hz)')
    plt.ylabel('Amplitude |X(freq)|')
    plt.title('Amplitude spectrum')
    plt.xlim(0.01, 0.09)

Using the `data_gaps` function defined above, we can now generate a plot that allows us to interactively vary the parameters $T_e$ and $t_e$.

In [ ]:
interactive_plot = interactive(data_gaps, Te=(0., 1000., 25.), te=(0., 1000., 100.))
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

In the next step, we will try to fill in the data gaps with interpolated values. To do this, we will load the `SciPy` function [interp1d](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.interp1d.html) ...

In [ ]:
# Load interpolation function
from scipy.interpolate import interp1d

... remove the data in the gap and interpolate the missing values ...

In [ ]:
def data_gaps_interp(Te, te, a):
    
    # Define parameters
    T = 25.                  # period [s]
    dt = .1                  # time sampling [s]
    L = 1000.                # length of the time series [s]

    # Define sine function with linear trend ...
    omega = 2. * np.pi / T   # compute circular frequency from period
    t = np.arange(0,L+dt,dt) # compute time vector
    x = np.sin(omega*t)      # compute sine wave
    xlin = a*t               # create linear trend
    x = x + xlin             # add linear trend    
    
    # Add data gap 
    NTeh = (int)((Te/2) // dt) # half size of the gap [#samples]
    Nte = (int)(te // dt)      # location of the gap [#samples]
    
    # Remove values inside the data gap
    xg = np.delete(x, np.arange(Nte-NTeh,Nte+NTeh))
    tg = np.delete(t, np.arange(Nte-NTeh,Nte+NTeh))    
        
    # Interpolate values in data gap
    xint = interp1d(tg, xg, 'cubic')  # create interpolation function
    x = xint(t)              # interpolate values for original time t
    
    # Fourier transform
    X = np.fft.fft(x)
    
    # Normalize by length of the time series
    N = len(x)               # length of the time series [#samples]
    X = X / (N*dt)
    
    # estimate frequencies    
    freq = np.fft.fftfreq(N, d=dt)
    
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    plt.subplot(121)
    
    plt.plot(t, x, 'b')
    plt.plot(t[Nte],0,'ro')   # mark center of data gap
    plt.xlabel('Time (s)')
    plt.ylabel('Sin(t) + a*t ()')
    plt.title('Time Series Sin(t) + a*t')
    
    # Plot Spectrum
    plt.subplot(122)
    
    plt.plot(freq, np.abs(X), 'b')
    plt.xlabel('Freq (Hz)')
    plt.ylabel('Amplitude |X(freq)|')
    plt.title('Amplitude spectrum')
    plt.xlim(0.0, 0.09)

In [ ]:
interactive_plot = interactive(data_gaps_interp, Te=(0., 1000., 25.), te=(0., 1000., 100.), a=(0.,1.,0.1))
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

The results of the interpolation are more or less successful - less so when the data gap becomes too large and the function too complex.

## Summary:

- Introducing data gaps results in local maxima in the amplitude spectrum
- By interpolating the data, we can fill the gap as long as it is not too large and the function not too complicated.
- However, the example also shows that the interpolated values should be interpreted with caution.